# Chapter 1.3 — Text-to-image search (SigLIP + FAISS)

Build the chapter’s multimodal mini-app:

1. Encode ~100 WikiArt images with **SigLIP**’s image tower.
2. Store the vectors in **FAISS**.
3. Encode a text query with the **text** tower.
4. Nearest neighbors = paintings that match the sentence.

This is CLIP math plus an index. Retrieval, ranking, and RAG-style multimodal search all start here.

**Colab:** GPU recommended. Add a Hugging Face secret named `HF_TOKEN` (read access).

**Dimension fix:** the early-release draft says SigLIP-base *and* 1152-D vectors. Those do not match. Base SigLIP is 768-D; 1152-D is the so400m family. We **read the size from the model** and pass that to FAISS.


## 0. Setup


In [ ]:
import os
import sys
import subprocess
import torch

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "accelerate",
        "sentencepiece",
        "pillow",
        "matplotlib",
        "requests",
        "huggingface_hub",
        "faiss-cpu",
    ]
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# Optional GPU FAISS; CPU index is fine for 100 vectors.
if DEVICE == "cuda":
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu-cu12"])
        print("installed faiss-gpu-cu12")
    except Exception as exc:
        print("faiss-gpu optional install skipped:", exc)

import faiss
import numpy as np
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from huggingface_hub import login

# Colab secret → env → already-logged-in CLI
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    from google.colab import userdata

    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    login(token=token)
    os.environ["HF_TOKEN"] = token
    print("logged in to Hugging Face")
else:
    print("No HF_TOKEN found. Public rows may still work; if this cell fails later, add a token.")


## 1. Load SigLIP and measure embedding size

`google/siglip-base-patch16-224` is the checkpoint the chapter names. We project images and text into the same vector space, then L2-normalize so FAISS L2 search ≈ cosine nearest neighbors.


In [ ]:
from transformers import AutoModel, AutoProcessor

MODEL_ID = "google/siglip-base-patch16-224"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

# Measure dim from an actual forward pass so FAISS always matches the checkpoint.
probe = Image.new("RGB", (224, 224), color=(128, 128, 128))
probe_inputs = processor(images=probe, return_tensors="pt")
probe_inputs = {k: v.to(DEVICE) for k, v in probe_inputs.items()}
with torch.no_grad():
    EMBED_DIM = int(model.get_image_features(**probe_inputs).shape[-1])

print("model", MODEL_ID)
print("embedding dim from get_image_features:", EMBED_DIM)
print("vision hidden size", getattr(model.config.vision_config, "hidden_size", "?"))
print("text hidden size", getattr(model.config.text_config, "hidden_size", "?"))


## 2. Stream 100 WikiArt rows

The chapter uses the Hub datasets-server HTTP API so you can preview rows (image URLs) without downloading the full dataset.


In [ ]:
API_URL = (
    "https://datasets-server.huggingface.co/rows"
    "?dataset=vlmbook/wikiart&config=default&split=train&offset=1&length=100"
)
headers = {}
tok = os.environ.get("HF_TOKEN")
if tok:
    headers["Authorization"] = f"Bearer {tok}"

resp = requests.get(API_URL, headers=headers, timeout=60)
resp.raise_for_status()
payload = resp.json()
rows = payload["rows"]
print("fetched", len(rows), "rows")
print("first row keys:", rows[0]["row"].keys())
print("sample image url:", rows[0]["row"]["image"]["src"][:120], "...")


def fetch_image(url: str, max_side: int = 384) -> Image.Image:
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert("RGB")
    w, h = img.size
    scale = max_side / max(w, h)
    if scale < 1:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return img


preview = fetch_image(rows[0]["row"]["image"]["src"])
plt.imshow(preview)
plt.axis("off")
plt.title("WikiArt sample")
plt.show()


## 3. Embed images and add them to FAISS

`IndexFlatL2` is brute-force L2. Perfect for 100 vectors. For millions of images you would switch to an approximate index; the embedding code would stay the same.


In [ ]:
def embed_images(images: list[Image.Image]) -> np.ndarray:
    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        vecs = model.get_image_features(**inputs)
        vecs = torch.nn.functional.normalize(vecs, dim=-1)
    return vecs.cpu().numpy().astype("float32")


def embed_text(texts: list[str]) -> np.ndarray:
    inputs = processor(text=texts, return_tensors="pt", padding=True)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        vecs = model.get_text_features(**inputs)
        vecs = torch.nn.functional.normalize(vecs, dim=-1)
    return vecs.cpu().numpy().astype("float32")


index = faiss.IndexFlatL2(EMBED_DIM)
urls = []
batch, batch_urls = [], []
BATCH = 8

for item in rows:
    url = item["row"]["image"]["src"]
    try:
        image = fetch_image(url)
    except Exception as exc:
        print("skip", url, exc)
        continue
    batch.append(image)
    batch_urls.append(url)
    if len(batch) == BATCH:
        index.add(embed_images(batch))
        urls.extend(batch_urls)
        batch, batch_urls = [], []
        print(f"indexed {index.ntotal} images", flush=True)

if batch:
    index.add(embed_images(batch))
    urls.extend(batch_urls)

print("FAISS ntotal", index.ntotal, "| dim", index.d, "| urls", len(urls))
assert index.d == EMBED_DIM
assert index.ntotal == len(urls)


## 4. Search with a text query

Because both towers are normalized, the smallest L2 distance is the largest cosine similarity.


In [ ]:
def search(query: str, k: int = 5):
    q = embed_text([query])
    distances, indices = index.search(q, k)
    hits = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < 0:
            continue
        hits.append((float(dist), urls[int(idx)]))
    return hits


def show_hits(query: str, k: int = 5):
    hits = search(query, k=k)
    fig, axes = plt.subplots(1, len(hits), figsize=(3 * len(hits), 3.2))
    if len(hits) == 1:
        axes = [axes]
    for ax, (dist, url) in zip(axes, hits):
        ax.imshow(fetch_image(url))
        ax.set_title(f"L2={dist:.3f}")
        ax.axis("off")
    plt.suptitle(repr(query))
    plt.tight_layout()
    plt.show()
    return hits


show_hits("a portrait of a person")
show_hits("a landscape with trees and sky")


Try your own prompt. If results look random, the index is too small or the query is outside WikiArt’s styles — that is a data limit, not a FAISS bug.


In [ ]:
# Edit this string and re-run.
show_hits("a still life with flowers")


## What you built

```
text query → SigLIP text tower → vector
                                  ↓
                         FAISS nearest neighbor
                                  ↓
image gallery ← SigLIP image tower ← WikiArt JPEGs
```

Closed-set ViT says “this ImageNet class.” CLIP/SigLIP say “this vector is near that sentence.” Search is the same comparison against a database instead of against four labels.

Chapter 2 will put similar models to work on captioning, VQA, and grounding. Chapter 3 explains attention inside VLMs. You already have the prerequisite: **a shared embedding space.**
